In [ ]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root aggiunta al path: {project_root}")

Project Root aggiunta al path: c:\Users\emagi\Documents\Deep_Learning\Progetto_deep_learning


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from src.Models.naive import NaivePersistence
from src.Training.engine import validate_one_epoch
from src.Data_loading.data_loader import TS_Cross_Validator
from src import config

# 1. Caricamento Dati
df = pd.read_csv("../data/processed/adjusted_ds.csv", index_col=0) 

# 2. Creazione Folds
validator = TS_Cross_Validator(df, config.TARGET_COL, config.SAMPLING_CONFIG) 
folds = validator.get_folds()

# 3. Setup Modello Naive
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

naive_model = NaivePersistence().to(device) 

# Usiamo L1Loss perché vogliamo calcolare il MAE per il denominatore del MASE
mae_metric = nn.L1Loss() 

naive_maes = []

print(f"--- Calcolo Benchmark Naive su {len(folds)} Fold ---\n")

for i, (train_loader, val_loader, scaler) in enumerate(folds):
    
    _, fold_mae = validate_one_epoch(naive_model, val_loader, nn.MSELoss(), device)
    
    naive_maes.append(fold_mae)
    print(f"FOLD {i+1} -> Naive MAE: {fold_mae:.6f}")

print("\n--- DA COPIARE NEL CONFIG.PY ---")
print(f"NAIVE_MAE_PER_FOLD = {naive_maes}")


=== INIZIO ISPEZIONE (3 splits) ===
Totale righe dataset: 17544

---------------- FOLD 0 ----------------
SHAPE -> Train: (4386, 26) | Val: (4386, 26)
TRAIN -> Da: 0  A: 4385
VAL   -> Da: 4386  A: 8771
Contiguità: Fine Train (4385) -> Inizio Val (4386)
Fold 1 creato -> Train samples: 4315, Val samples: 4363

---------------- FOLD 1 ----------------
SHAPE -> Train: (8772, 26) | Val: (4386, 26)
TRAIN -> Da: 0  A: 8771
VAL   -> Da: 8772  A: 13157
Contiguità: Fine Train (8771) -> Inizio Val (8772)
Fold 2 creato -> Train samples: 8701, Val samples: 4363

---------------- FOLD 2 ----------------
SHAPE -> Train: (13158, 26) | Val: (4386, 26)
TRAIN -> Da: 0  A: 13157
VAL   -> Da: 13158  A: 17543
Contiguità: Fine Train (13157) -> Inizio Val (13158)
Fold 3 creato -> Train samples: 13087, Val samples: 4363
--- Calcolo Benchmark Naive su 3 Fold ---

FOLD 1 -> Naive MAE: 0.061885
FOLD 2 -> Naive MAE: 0.073407
FOLD 3 -> Naive MAE: 0.067460

--- COPIA QUESTO NEL TUO CONFIG.PY ---
NAIVE_MAE_PER_FOLD 